# Perspectives and visualizations

Europe’s energy landscape is undergoing a rapid, necessary transformation. The dual challenges of climate urgency and economic reality frame this important shift. Fully decarbonizing the continent’s energy system demands immense investment, estimated at trillions of euros (European Commission, 2020), even as renewable energy technologies become increasingly cost-competitive (IRENA, 2021). Our analysis delves into how Europe is navigating these complexities, powered by insights from recent data.

## The Economic and Social Feasibility Challenge

One critical aspect is understanding whether European countries can economically and socially handle this profound change. That requires examining not just current commitments, but the long shadow of the past. The sheer scale of historical reliance on fossil fuels is made visible through cumulative emissions: a legacy that still defines national responsibilities today. The chart below, based on emissions data from the Our World in Data CO2 dataset (OWID, 2023), shows how different European countries contributed to carbon buildup over time.

In [16]:
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

In [17]:
import pandas as pd
import plotly.express as px

# Data prep (same as before)
owid_co2_data_df = pd.read_csv('owid-co2-data.csv')
european_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
    'Estonia','Finland','France','Germany','Greece','Hungary','Ireland',
    'Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden',
    'United Kingdom','Norway','Switzerland','Iceland'
]
df_cumulative = owid_co2_data_df[
    (owid_co2_data_df['country'].isin(european_countries)) &
    (owid_co2_data_df['year'] >= 1950)
].copy()
df_cumulative = df_cumulative[['country','year','cumulative_co2']].dropna()
df_cumulative['cumulative_co2'] = pd.to_numeric(df_cumulative['cumulative_co2'])

contrast_colors = px.colors.qualitative.Bold

fig = px.area(
    df_cumulative,
    x="year",
    y="cumulative_co2",
    color="country",
    line_group="country",
    hover_name="country",
    title="Cumulative CO₂ Emissions from Fossil Fuels\nin European Countries (since 1950)",
    labels={"year":"Year","cumulative_co2":"Cumulative CO₂ Emissions (Mt)"},
    color_discrete_sequence=contrast_colors,
    line_shape='spline'
)
for trace in fig.data:
    trace.line.smoothing = 1.3

# Region filter (same as before)
west  = ['Belgium','France','Ireland','Luxembourg','Netherlands','United Kingdom']
south = ['Croatia','Cyprus','Greece','Italy','Malta','Portugal','Spain']
north = ['Denmark','Estonia','Finland','Iceland','Latvia','Lithuania','Norway','Sweden']
east  = ['Austria','Bulgaria','Czechia','Germany','Hungary','Poland','Romania','Slovakia','Slovenia','Switzerland']
trace_names = [t.name for t in fig.data]
buttons = [
    dict(label='All Europe',      method='update', args=[{'visible': [True]*len(trace_names)}, {'title':'All Europe'}]),
    dict(label='Western Europe',  method='update', args=[{'visible': [name in west for name in trace_names]}, {'title':'Western Europe'}]),
    dict(label='Southern Europe', method='update', args=[{'visible': [name in south for name in trace_names]}, {'title':'Southern Europe'}]),
    dict(label='Northern Europe', method='update', args=[{'visible': [name in north for name in trace_names]}, {'title':'Northern Europe'}]),
    dict(label='Eastern Europe',  method='update', args=[{'visible': [name in east for name in trace_names]}, {'title':'Eastern Europe'}])
]
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.15, y=1.15,
        xanchor='right', yanchor='top'
    )],
    legend=dict(
        title='Country',
        itemclick='toggle',
        itemdoubleclick='toggleothers'
    ),
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    margin=dict(l=60, r=30, t=70, b=120),  # Extra bottom margin for annotation
)

fig.update_xaxes(
    showgrid=True, gridcolor='rgba(255,255,255,0.2)',
    zerolinecolor='white', tickfont_color='white', title_font_color='white'
)
fig.update_yaxes(
    showgrid=False, tickfont_color='white', title_font_color='white'
)

# --- The key: annotation anchored inside the plot! ---
fig.add_annotation(
    text=(
        "Click countries in the legend to toggle them on/off.<br>"
        "Use the filter dropdown (top-right) to show Western, Southern, Northern or Eastern Europe.<br>"
        "Hover over any line to see detailed yearly values."
    ),
    xref="paper", yref="paper",
    x=-00.05, y=-0.33,  # y<0 puts it below the x-axis, but inside the figure!
    showarrow=False,
    font=dict(color='white', size=10),
    align='left'
)

fig.show(config={'responsive': True})





As the visualization shows, nations like the United Kingdom and Germany have emitted far more than others over the last seven decades. Countries such as France, Italy, Poland, and Spain follow, but at lower cumulative levels. This pattern reinforces the idea that transition pressures are not equal—those who polluted more in the past now face a greater moral and logistical burden to decarbonize fast. The historical imbalance is a central part of why transition strategies differ so widely across Europe.


Financial readiness also plays a decisive role. Moving away from fossil fuels isn't just an environmental challenge—it’s an economic one. Large-scale shifts require major capital flows into new infrastructure, subsidies, and system upgrades. Using production and value data from the IEA Monthly Electricity Statistics (2010–2025), the following chart tracks energy-related expenditures across major European economies.

In [18]:
import pandas as pd
import plotly.express as px

# 1) Data laden en bewerken
price = pd.read_csv('european_wholesale_electricity_price_data_monthly.csv', parse_dates=['Date'])
gen = pd.read_csv('data2.csv', skiprows=7,
                  names=['Country','Time','Balance','Product','Value','Unit'],
                  low_memory=False)

gen = gen[gen['Balance']=='Net Electricity Production'].copy()
gen['Time'] = pd.to_datetime(gen['Time'], format='%B %Y')
gen['Value'] = pd.to_numeric(gen['Value'], errors='coerce')

total_prod = gen[gen['Product']=='Electricity'][['Country','Time','Value']].rename(columns={'Value':'Total_GWh'})
renew_prod = gen[gen['Product']=='Total Renewables (Hydro, Geo, Solar, Wind, Other)'][['Country','Time','Value']].rename(columns={'Value':'Renewable_GWh'})

df = total_prod.merge(renew_prod, on=['Country','Time'], how='inner')
df = df.merge(
    price[['Country','Date','Price (EUR/MWhe)']].rename(columns={'Date':'Time','Price (EUR/MWhe)':'Price'}),
    on=['Country','Time'], how='inner'
)

df['Total_MWh']     = df['Total_GWh'] * 1000
df['Renewable_MWh'] = df['Renewable_GWh'] * 1000
df['Cost_Total']    = df['Price'] * df['Total_MWh']
df['Cost_Renewable']= df['Price'] * df['Renewable_MWh']
df['Cost_Fossil']   = df['Cost_Total'] - df['Cost_Renewable']

df['Year'] = df['Time'].dt.year
agg = df.groupby(['Year','Country'])[['Cost_Renewable','Cost_Fossil','Cost_Total']].sum() / 1e9
agg = agg.reset_index()

# 2) Fixed vertical order gebaseerd op laatste jaar
last_year   = agg['Year'].max()
fixed_order = (
    agg[agg['Year']==last_year]
    .sort_values('Cost_Total', ascending=False)['Country']
    .tolist()
)

# 3) Maak animated bar chart met fixed order, Renewable in groen
fig = px.bar(
    agg,
    x=['Cost_Renewable','Cost_Fossil'],
    y='Country',
    orientation='h',
    animation_frame='Year',
    category_orders={'Year': sorted(agg['Year'].unique()), 'Country': fixed_order},
    labels={'value':'Cost (billion EUR)','variable':'Type','Country':'Country','Year':'Year'},
    title='Energy Costs per European Country (Renewable vs Fossil)',
    color_discrete_map={'Cost_Renewable':'#2CA02C'}  # Renewable in groen
)

fig.update_layout(
    barmode='stack',
    margin=dict(l=50, r=20, t=50, b=130),  # <--- Only this is changed!
    transition={'duration':1000,'easing':'cubic-in-out'}
)

# 4) Smooth slider
if fig.layout.sliders:
    fig.layout.sliders[0].transition = {'duration':1000,'easing':'cubic-in-out'}

# 5) Fixed reversed y-axis
fig.update_yaxes(autorange='reversed', categoryorder='array', categoryarray=fixed_order)

# 6) Definieer per-jaar x-as limieten
year_max = {
    **{yr: 30  for yr in range(2015, 2021)},
    **{yr: 130 for yr in (2021,2022,2023)},
    2025: 30
}

# 7) Pas per-frame x-as en forceer redraw
for frame in fig.frames:
    yr = int(frame.name)
    if yr in year_max:
        frame.layout.xaxis.range     = [0, year_max[yr]]
        frame.layout.xaxis.autorange = False
    else:
        frame.layout.xaxis.autorange = True
    frame.layout.yaxis = {'autorange':'reversed'}

updater = fig.layout.updatemenus[0].buttons[0].args[1]
updater['frame']['redraw']      = True
updater['transition']['duration']= 1000
updater['transition']['easing']  = 'cubic-in-out'
updater['frame']['duration']     = 2000
updater['frame']['easing']       = 'cubic-in-out'

# 8) Optioneel: forceer basislayout op langst geldende max
global_max = max(year_max.values())
fig.update_xaxes(range=[0, global_max], autorange=False, rangemode='tozero')

fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084'
)

# As- en gridlijnen wit
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.2)', zerolinecolor='white', color='white')
fig.update_yaxes(showgrid=False, color='white')

# Titel- en legend-teksten wit
fig.update_layout(
    title_font_color='white',
    legend_font_color='white'
)

# Slider en knoppen styling
if fig.layout.sliders:
    s = fig.layout.sliders[0]
    s.bgcolor      = '#004084'
    s.bordercolor  = 'white'
    s.borderwidth  = 1
    s.font.color   = 'white'
    s.currentvalue.font.color = 'white'

if fig.layout.updatemenus:
    u = fig.layout.updatemenus[0]
    u.bgcolor     = '#004084'
    u.bordercolor = 'white'
    u.borderwidth = 1
    u.font.color  = 'white'

# ... your code above unchanged ...
fig.update_layout(
    barmode='stack',
    margin=dict(l=50, r=20, t=50, b=200),  # b=200 for lots of blue space!
    transition={'duration':1000,'easing':'cubic-in-out'}
)
# ... code ...
fig.add_annotation(
    text=(
        "Click Cost_Renewable or Cost_Fossil in the legend to toggle them on/off.<br>"
        "Click the play/stop button to start/stop the animation. Use the slider to select a certain year.<br>"
        "Hover over any bar to see detailed yearly cost values of the country."
    ),
    xref="paper", yref="paper",
    x=-0.15, y=-0.70,   # try -0.75 if you need it even lower!
    showarrow=False,
    font=dict(color='white', size=10),
    align='left'
)


fig.show(config={'responsive': True})







The graph highlights how countries like France, Germany, the UK, Italy, and Spain saw energy costs rise steeply, particularly in 2022, with some surpassing €100 billion. This spike aligns with the post-pandemic demand surge and market shocks stemming from the Russia–Ukraine conflict (Cevik et al., 2022). By 2025, costs fall again, but the volatility illustrates just how sensitive these investments are to global disruptions. Managing these fiscal shifts remains a key challenge for policy stability and public acceptance.


The economic turbulence seen in national investment patterns is echoed in electricity markets. Consumers across the continent experienced dramatic fluctuations in wholesale electricity prices, captured in monthly data from Ember (2015–2025). This price movement has both short-term impacts—on affordability—and long-term implications for investor confidence in clean energy.

In [19]:
import pandas as pd 
import plotly.express as px
import plotly.graph_objects as go  # toegevoegd voor extra trace

# Data inladen en filteren
df = pd.read_csv('european_wholesale_electricity_price_data_monthly.csv', parse_dates=['Date'])
european_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
    'Estonia','Finland','France','Germany','Greece','Hungary','Ireland',
    'Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden',
    'United Kingdom','Norway','Switzerland','Iceland'
]
df = df[df['Country'].isin(european_countries)].dropna(subset=['Price (EUR/MWhe)'])

# Basis line chart met per-land lijnen
fig = px.line(
    df,
    x='Date',
    y='Price (EUR/MWhe)',
    color='Country',
    title='Electricity Price Trends in Selected European Countries',
    labels={'Price (EUR/MWhe)':'Price (EUR/MWhe)','Date':'Date'},
)

# 1) Voeg Europese trendlijn (gemiddelde) toe in zwart
avg_df = df.groupby('Date')['Price (EUR/MWhe)'].mean().reset_index()
fig.add_trace(
    go.Scatter(
        x=avg_df['Date'],
        y=avg_df['Price (EUR/MWhe)'],
        mode='lines',
        line=dict(color='black', width=4),
        name='Europe Average'
    )
)

# 2) Zet Europe Average bovenaan in de legend
all_traces   = list(fig.data)
avg_trace    = [t for t in all_traces if t.name == 'Europe Average'][0]
other_traces = [t for t in all_traces if t.name != 'Europe Average']
fig.data      = tuple([avg_trace] + other_traces)

# 3) Definieer regio’s voor dropdown-filter
west  = ['Belgium','France','Ireland','Luxembourg','Netherlands','United Kingdom']
south = ['Croatia','Cyprus','Greece','Italy','Malta','Portugal','Spain']
north = ['Denmark','Estonia','Finland','Iceland','Latvia','Lithuania','Norway','Sweden']
east  = ['Bulgaria','Czechia','Hungary','Poland','Romania','Slovakia','Slovenia']

# 4) Maak lijst met alle trace-namen (inclusief Europe Average)
trace_names = [t.name for t in fig.data]

# 5) Buttons inclusief “Europe Average Only”
buttons = [
    dict(label='All Countries',
         method='update',
         args=[{'visible': [True]*len(trace_names)}, {}]),
    dict(label='Europe Average Only',
         method='update',
         args=[{'visible': [name=='Europe Average' for name in trace_names]},
               {'title':'Europe Average Only'}]),
    dict(label='Western Europe',
         method='update',
         args=[{'visible': [name in west or name=='Europe Average' for name in trace_names]},
               {'title':'Western Europe'}]),
    dict(label='Southern Europe',
         method='update',
         args=[{'visible': [name in south or name=='Europe Average' for name in trace_names]},
               {'title':'Southern Europe'}]),
    dict(label='Northern Europe',
         method='update',
         args=[{'visible': [name in north or name=='Europe Average' for name in trace_names]},
               {'title':'Northern Europe'}]),
    dict(label='Eastern Europe',
         method='update',
         args=[{'visible': [name in east  or name=='Europe Average' for name in trace_names]},
               {'title':'Eastern Europe'}])
]

# 6) Voeg dropdown en legend-click functionaliteit toe, en marge onder text
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.15, y=1.15,
        xanchor='right', yanchor='top'
    )],
    legend=dict(
        title='Country',
        itemclick='toggle',
        itemdoubleclick='toggleothers'
    ),
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    margin=dict(l=50, r=50, t=80, b=200)  # increased bottom margin
)

# 7) Styling assen en grid
fig.update_xaxes(
    showgrid=True, gridcolor='rgba(255,255,255,0.2)',
    tickfont_color='white', title_font_color='white'
)
fig.update_yaxes(
    showgrid=True, gridcolor='rgba(255,255,255,0.2)',
    tickfont_color='white', title_font_color='white'
)

# 8) Onder de grafiek instructietekst in het Engels toevoegen, lager geplaatst
fig.update_layout(
    annotations=[dict(
        text="Click a country in the legend to toggle individual lines.<br>"
             "Use the dropdown at top right to quickly filter Western, Eastern, Northern, or Southern Europe.<br>"
             "Zoom in by right clicking and making a square.",
        xref='paper', yref='paper',
        x=-0.03, y=-0.3,   # moved down for clear separation
        showarrow=False,
        font=dict(color='white', size=10),
        align='left'
    )]
)

fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    title_font_size=24,
    legend_title_text='Country',
    margin=dict(l=50, r=30, t=70, b=125)  # increase 'b' (bottom) from default 50 to e.g. 120
)

fig.show(config={'responsive': True})











Electricity prices remained relatively stable under €100/MWh until 2021. Then, mid-2022 saw a sudden spike, with prices in Italy and others reaching over €500/MWh, before dropping again by early 2023. This mirrors the expenditure surge in the previous graph and reinforces how energy transitions can amplify market instability. The same geopolitical disruptions identified earlier (Cevik et al., 2022) also shaped consumer markets, reminding us that energy systems do not transform in isolation from broader global events.

At the national level, a country’s economic capacity heavily influences how it manages this transition. But emissions are also tied to wealth: wealthier countries tend to produce more carbon, historically and in the present. Using the 2022 data from OWID’s emissions and GDP records, the following scatterplot illustrates this relationship across the EU.

In [20]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go  # voor de regressielijn

# 1) Data inladen en filteren
url = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'
df = pd.read_csv(url)
df = df[df['year'] == 2022]
eu_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia',
    'Denmark','Estonia','Finland','France','Germany','Greece',
    'Hungary','Ireland','Italy','Latvia','Lithuania','Luxembourg',
    'Malta','Netherlands','Poland','Portugal','Romania',
    'Slovakia','Slovenia','Spain','Sweden'
]
df = df[df['country'].isin(eu_countries)].dropna(subset=['gdp','co2'])

# 2) Log-transform
df['log_gdp'] = np.log10(df['gdp'])
df['log_co2'] = np.log10(df['co2'])

# 3) Eenschriftige y-schaal: elke 0.5 wordt 1 plot-eenheid
scale = 1/0.5
df['y_plot'] = df['log_co2'] * scale

# 4) Originele y-ticks van bijv. -1.0 tot max per 0.5
y_min, y_max = df['log_co2'].min(), df['log_co2'].max()
orig_min = np.floor(y_min*2)/2
orig_max = np.ceil(y_max*2)/2
orig_ticks = np.arange(orig_min, orig_max + 0.5, 0.5)
plot_ticks = list(orig_ticks * scale)
plot_labels = [f"{t:.1f}" for t in orig_ticks]

# 5) Bereken OLS-regressie over alle landen
slope, intercept = np.polyfit(df['log_gdp'], df['log_co2'], 1)
x_line = np.linspace(df['log_gdp'].min(), df['log_gdp'].max(), 100)
y_line = slope * x_line + intercept
y_line_plot = y_line * scale  # zelfde schaaltransformatie

# 6) Maak scatter met kleur per land
fig = px.scatter(
    df,
    x='log_gdp',
    y='y_plot',
    color='country',
    hover_name='country',
    title='Correlation GDP vs CO₂ Emissions (EU, 2022) — log scale',
    labels={'log_gdp':'Log₁₀ GDP (dollars)', 'y_plot':'Log₁₀ CO₂ (ton)'},
)

# 7) Voeg de regressielijn toe als witte lijn
fig.add_trace(
    go.Scatter(
        x=x_line,
        y=y_line_plot,
        mode='lines',
        line=dict(color='white', width=2),
        name='Trendline (OLS)'
    )
)

# 8) Styling markers
fig.update_traces(
    marker=dict(size=8, line=dict(width=1)),
    selector=lambda tr: 'markers' in tr.mode
)

# 9) Achtergrond & font kleuren
fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    title_font_size=24,
    legend_title_text='Country'
)

# 10) As-instellingen
fig.update_xaxes(
    tickmode='linear',
    dtick=0.5,
    showgrid=True,
    gridcolor='rgba(255,255,255,0.3)',
    zerolinecolor='white',
    tickfont_color='white',
    title_font_color='white'
)
fig.update_yaxes(
    tickmode='array',
    tickvals=plot_ticks,
    ticktext=plot_labels,
    range=[plot_ticks[0], plot_ticks[-1]],
    autorange=False,
    showgrid=True,
    gridcolor='rgba(255,255,255,0.3)',
    zerolinecolor='white',
    tickfont_color='white',
    title_font_color='white'
)

fig.add_annotation(
    text=(
        "Click on the countries in the legend to toggle them on/off.<br>"
        "Hover over the points on the graph to see more details about the point.<br>"
        "Zoom in by right clicking and making a square."
        
    ),
    xref="paper", yref="paper",
    x=-0.03, y=-0.25,   # try -0.75 if you need it even lower!
    showarrow=False,
    font=dict(color='white', size=10),
    align='left'
)

fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    title_font_size=24,
    legend_title_text='Country',
    margin=dict(l=50, r=30, t=70, b=120)  # increase 'b' (bottom) from default 50 to e.g. 120
)

fig.show(config={'responsive': True})









Germany, France, Spain, and Italy occupy the top-right of the distribution—wealthy and high-emitting. Meanwhile, smaller economies such as Austria, Greece, and Slovakia show lower values on both axes. The implication is twofold: richer countries have contributed more to the problem, but they also hold the financial tools needed to lead the transition. Aligning responsibility with capability is a central political tension in EU climate planning (Apeti et al., 2025).

Despite disparities in wealth, many countries are scaling up renewable energy. Some benefit from geographic advantages like hydropower or longstanding policy frameworks. Drawing again from the IEA Monthly Electricity Statistics (March 2025), the next graph captures each country’s share of renewables in total electricity production.

In [21]:
import pandas as pd
import plotly.express as px

# --- Data inladen en bewerken ---
owid = pd.read_csv('owid-co2-data.csv')
prod = pd.read_csv('data2.csv', skiprows=8)

european_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
    'Estonia','Finland','France','Germany','Greece','Hungary','Ireland',
    'Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden',
    'United Kingdom','Norway','Switzerland','Iceland'
]

# Filter en periode
prod = prod[(prod['Balance']=='Net Electricity Production') &
            (prod['Unit']=='GWh') &
            (prod['Country'].isin(european_countries))].copy()
prod['Date_Parsed'] = pd.to_datetime(prod['Time'], format='%B %Y', errors='coerce')
latest = prod['Date_Parsed'].max()
prod = prod[prod['Date_Parsed']==latest]

# Opbouwen shares
renew = prod[prod['Product'].isin(['Hydro','Solar','Wind','Biofuels','Geothermal','Other renewables'])]
tot   = prod[prod['Product']=='Electricity']
df_ren = renew.groupby('Country')['Value'].sum().rename('Renewable_GWh')
df_tot = tot  .groupby('Country')['Value'].sum().rename('Total_GWh')
df = pd.concat([df_ren, df_tot], axis=1).dropna()
df['Share'] = df['Renewable_GWh']/df['Total_GWh']*100
df = df.reset_index()

# ISO codes join
iso = owid[['country','iso_code']].dropna().drop_duplicates()
df = df.merge(iso, left_on='Country', right_on='country', how='left').dropna(subset=['iso_code'])

# --- Verticale bar chart ---
df_sorted = df.sort_values('Share', ascending=False)  # grootste eerst

fig = px.bar(
    df_sorted,
    x='Country',
    y='Share',
    color='Share',
    color_continuous_scale='Greens',
    title=f'Renewable Electricity Share by Country – {latest.strftime("%B %Y")}',
    labels={'Share':'Renewable Share (%)','Country':''},
)

# Styling
fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    title_font_size=24,
    margin=dict(l=80, r=50, t=80, b=150)
)
fig.update_yaxes(
    range=[0,100],
    gridcolor='rgba(255,255,255,0.2)',
    zerolinecolor='white',
    tickfont_color='white',
    title_font_color='white'
)
fig.update_xaxes(
    tickfont_color='white',
    tickangle=45
)

fig.add_annotation(
    text=(
        "Hover over the bar graph to see the exact renewable energy share of the country.<br>"
        "Zoom in by right clicking a square.<br>"
        
    ),
    xref="paper", yref="paper",
    x=-0.08, y=-0.40,   # try -0.75 if you need it even lower!
    showarrow=False,
    font=dict(color='white', size=10),
    align='left'
)



fig.show(config={'responsive': True})




Iceland and Norway are near 100% renewable, followed by Portugal, Luxembourg, and Croatia at around 80%. In contrast, major economies like France, Italy, the Netherlands, and Germany remain under 50%, underscoring how legacy infrastructure and fossil dependency still slow progress. These gaps reflect not just political will, but deep-seated differences in energy systems, natural resource access, and financial flexibility.

## The Climate Urgency Imperative

Beyond economic feasibility, the transition is driven by climate necessity. Decarbonizing is not just a budgetary decision—it’s essential to stop the warming that is already reshaping Europe. The chart below uses data from OWID (2000–2023) to show how annual CO2 emissions remain high across major economies.


In [22]:
import pandas as pd
import plotly.graph_objects as go

# 1) Data inladen en filteren
df = pd.read_csv('owid-co2-data.csv')
european_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
    'Estonia','Finland','France','Germany','Greece','Hungary','Ireland',
    'Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden',
    'United Kingdom','Norway','Switzerland','Iceland'
]
df = df[df['country'].isin(european_countries)].dropna(subset=['co2'])

# 2) Filter op jaren (bijv. vanaf 2000)
start_year = 2000
years = sorted(df['year'].unique())
years = [y for y in years if y >= start_year]
df = df[df['year'].isin(years)]

# 3) Pivot voor heatmap: landen × jaren
heatmap_df = df.pivot(index='country', columns='year', values='co2')
heatmap_df = heatmap_df.reindex(sorted(heatmap_df.index))  # optioneel: alfabetisch

# 4) Bouw Figure met clear cell boundaries
fig = go.Figure(data=go.Heatmap(
    z=heatmap_df.values,
    x=heatmap_df.columns,
    y=heatmap_df.index,
    colorscale='RdYlGn_r',
    colorbar=dict(title='CO₂ Emissions (Mt)'),
    hovertemplate="Country: %{y}<br>Year: %{x}<br>CO₂: %{z:.2f} Mt<extra></extra>"
))

# 5) Vakjes duidelijk maken
fig.update_traces(xgap=1, ygap=1)

# 6) Axis ticks voor elk jaar en land
fig.update_xaxes(
    tickmode='array',
    tickvals=heatmap_df.columns,
    ticktext=heatmap_df.columns,
    tickangle=45,
    title_text='Year',
    tickfont_color='white'
)
fig.update_yaxes(
    tickmode='array',
    tickvals=heatmap_df.index,
    ticktext=heatmap_df.index,
    title_text='Country',
    tickfont_color='white'
)

# 7) Layout & styling (background blue, title in English)
fig.update_layout(
    title='Annual CO₂ Emissions per European Country (2000–' + str(years[-1]) + ')',
    margin=dict(l=150, r=50, t=80, b=150),
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
)

fig.add_annotation(
    text=(
        "Hover over the heat map to see the exact CO2 emmissions of each country from 2000-2023.<br>"
        "Zoom in by right clicking a square.<br>"
        
    ),
    xref="paper", yref="paper",
    x=-0.20, y=-0.40,   # try -0.75 if you need it even lower!
    showarrow=False,
    font=dict(color='white', size=10),
    align='left'
)
# 8) Show responsive for GitHub Pages
fig.show(config={'responsive': True})






Germany remains Europe’s top emitter, followed by France, Poland, and the UK, though all have reduced their emissions significantly since 2000. For instance, Germany dropped from nearly 900 million tonnes to below 600. These declines matter—but absolute numbers remain enormous. The urgency remains clear: despite progress, current emissions still drive global warming and ecological instability.

To understand relative ambition, we need to see how countries differ in their emission reductions. OWID’s data on percentage change from 2000 to 2023 highlights uneven progress across Europe. Some nations have cut emissions sharply, while others have increased.


In [23]:
import pandas as pd
import plotly.express as px

# 1) Load data
df = pd.read_csv('owid-co2-data.csv')

# 2) Filter for European countries
european_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
    'Estonia','Finland','France','Germany','Greece','Hungary','Ireland',
    'Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden',
    'United Kingdom','Norway','Switzerland','Iceland'
]
df_europe = df[df['country'].isin(european_countries)].copy()

# 3) Extract 2000 and 2023 data
df_2000 = df_europe[df_europe['year'] == 2000][['country', 'co2']].set_index('country')
df_2023 = df_europe[df_europe['year'] == 2023][['country', 'co2']].set_index('country')

# 4) Combine and calculate percentage change
combined_df = df_2000.join(df_2023, lsuffix='_2000', rsuffix='_2023')
combined_df['percentage_change'] = ((combined_df['co2_2023'] - combined_df['co2_2000']) / combined_df['co2_2000']) * 100

# Reset index to make 'country' a column again for Plotly
percentage_change_df = combined_df.reset_index()

# Filter out countries with NaN in percentage_change (missing data for 2000 or 2023)
percentage_change_df.dropna(subset=['percentage_change'], inplace=True)

# 5) Create the treemap (make it bigger!)
fig = px.treemap(
    percentage_change_df,
    path=[px.Constant("European Countries"), 'country'],
    values='co2_2023',
    color='percentage_change',
    color_continuous_scale='RdYlGn_r',
    color_continuous_midpoint=0,
    title='Percentage Change in CO₂ Emissions (2000 - 2023) in European Countries',
    labels={'percentage_change': 'Percentage Change (%)', 'co2_2023': '2023 CO₂ Emissions (Mt)'},
    hover_data={'percentage_change': ':.2f%'},
    width=1050,     # <-- Wider
    height=750      # <-- Taller for more space for the graph and annotation
)

fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    margin=dict(l=50, r=40, t=70, b=100)  # <-- Bottom margin bigger for text space
)

# Customizing the hover template
fig.update_traces(
    hovertemplate="<b>%{label}</b><br>" +
                  "Percentage Change: %{color:.2f}%<br>" +
                  "2023 CO₂: %{value:.2f} Mt<extra></extra>"
)

# Add your annotation a little lower (change y to -0.25 if you want it even further down)
fig.add_annotation(
    text=(
        "Click on a country square to compare your selected country with all of Europe.<br>"
        "Hover over squares to see the detailed percentage change from 2000 to 2023."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.10,
    showarrow=False,
    font=dict(color='white', size=10),
    align='left'
)

fig.show()


Denmark, Portugal, and the Netherlands stand out for reductions of 30% to nearly 50%. Meanwhile, Iceland (+28.26%) and Lithuania (+4.92%) show emission increases. These patterns highlight that not all transition pathways are equally effective—and that lagging countries may need targeted support or stronger incentives to catch up.

Finally, historical emissions correlate directly with one of the transition’s key motivators: temperature rise. To illustrate this link, we combine OWID’s cumulative CO2 data with Berkeley Earth’s historical land temperature records (1900–2013) to plot environmental change over time.

In [24]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import linregress # For calculating trendlines

# Define European countries (copied from previous successful runs)
european_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
    'Estonia','Finland','France','Germany','Greece','Hungary','Ireland',
    'Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden',
    'United Kingdom','Norway','Switzerland','Iceland'
]

# Load and prepare CO2 data (copied from previous successful runs)
df_co2 = pd.read_csv('owid-co2-data.csv')
df_co2_filtered = df_co2[df_co2['country'].isin(european_countries)][['country', 'year', 'co2']].copy()

# Load and prepare Temperature data (copied from previous successful runs)
df_temp = pd.read_csv('GlobalLandTemperaturesByCountry.csv')
df_temp['dt'] = pd.to_datetime(df_temp['dt'])
df_temp['year'] = df_temp['dt'].dt.year
df_temp.rename(columns={'Country': 'country'}, inplace=True)
df_temp_yearly = df_temp.groupby(['country', 'year'])['AverageTemperature'].mean().reset_index()
df_temp_filtered = df_temp_yearly[df_temp_yearly['country'].isin(european_countries)].copy()

# Determine the common overlapping year range for both datasets
start_plot_year = 1900 # User requested start from 1900
end_plot_year = max(df_co2_filtered['year'].max(), df_temp_filtered['year'].max()) # End with the latest year from either dataset (2023)

# Calculate total European CO2 emissions per year and then cumulative CO2
europe_total_co2 = df_co2_filtered.groupby('year')['co2'].sum().reset_index()
europe_total_co2 = europe_total_co2[(europe_total_co2['year'] >= start_plot_year) & (europe_total_co2['year'] <= end_plot_year)].copy()
europe_total_co2['cumulative_co2'] = europe_total_co2['co2'].cumsum()
europe_total_co2.rename(columns={'co2': 'annual_co2_emissions'}, inplace=True) # Rename for clarity if needed

# Calculate European average annual temperature
europe_avg_temp = df_temp_filtered.groupby('year')['AverageTemperature'].mean().reset_index()
europe_avg_temp = europe_avg_temp[(europe_avg_temp['year'] >= start_plot_year) & (europe_avg_temp['year'] <= end_plot_year)].copy()
europe_avg_temp.rename(columns={'AverageTemperature': 'annual_avg_temp'}, inplace=True)

# Merge the cumulative CO2 and average annual temperature data
# Use 'outer' merge to keep all years in the range even if one has NaNs
df_plot_data = pd.merge(europe_total_co2, europe_avg_temp, on='year', how='outer')

# Filter for the exact plot range again in case outer merge brought in years outside it
df_plot_data = df_plot_data[(df_plot_data['year'] >= start_plot_year) & (df_plot_data['year'] <= end_plot_year)].copy()

# Drop rows with NaN values for actual calculations to avoid issues with linregress if any
# However, for plotting lines, NaNs will cause breaks. Linregress needs non-NaN data.
# So, we'll get non-null data for linregress separately.

# Calculate trendlines using linear regression for both variables
# For Cumulative CO2 - use non-null data for linregress
co2_data_for_trend = df_plot_data.dropna(subset=['cumulative_co2'])
if not co2_data_for_trend.empty:
    slope_co2, intercept_co2, r_value_co2, p_value_co2, std_err_co2 = linregress(co2_data_for_trend['year'], co2_data_for_trend['cumulative_co2'])
    df_plot_data['co2_trend'] = slope_co2 * df_plot_data['year'] + intercept_co2
else:
    df_plot_data['co2_trend'] = None # No trend if no data

# For Annual Average Temperature - use non-null data for linregress
temp_data_for_trend = df_plot_data.dropna(subset=['annual_avg_temp'])
if not temp_data_for_trend.empty:
    slope_temp, intercept_temp, r_value_temp, p_value_temp, std_err_temp = linregress(temp_data_for_trend['year'], temp_data_for_trend['annual_avg_temp'])
    df_plot_data['temp_trend'] = slope_temp * df_plot_data['year'] + intercept_temp
else:
    df_plot_data['temp_trend'] = None # No trend if no data

# Create the dual-axis line chart
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add Cumulative CO2 trace
fig.add_trace(
    go.Scatter(
        x=df_plot_data['year'],
        y=df_plot_data['cumulative_co2'],
        mode='lines',
        name='Europe Cumulative CO2 Emissions',
        line=dict(color='#AEC6CF', width=3), # Light blue for CO2
        showlegend=True
    ),
    secondary_y=False,
)

# Add Cumulative CO2 Trendline
# Plot only if trend data exists
if 'co2_trend' in df_plot_data.columns and df_plot_data['co2_trend'].notna().any():
    fig.add_trace(
        go.Scatter(
            x=df_plot_data['year'],
            y=df_plot_data['co2_trend'],
            mode='lines',
            name='CO2 Trendline',
            line=dict(color='#AEC6CF', width=2, dash='dot'), # Dotted trendline
            showlegend=True
        ),
        secondary_y=False,
    )

# Add Annual Average Temperature trace
fig.add_trace(
    go.Scatter(
        x=df_plot_data['year'],
        y=df_plot_data['annual_avg_temp'],
        mode='lines',
        name='Europe Annual Average Temperature',
        line=dict(color='#FF6347', width=3), # Tomato red for Temperature
        showlegend=True
    ),
    secondary_y=True,
)

# Add Annual Average Temperature Trendline
# Plot only if trend data exists
if 'temp_trend' in df_plot_data.columns and df_plot_data['temp_trend'].notna().any():
    fig.add_trace(
        go.Scatter(
            x=df_plot_data['year'],
            y=df_plot_data['temp_trend'],
            mode='lines',
            name='Temperature Trendline',
            line=dict(color='#FF6347', width=2, dash='dot'), # Dotted trendline
            showlegend=True
        ),
        secondary_y=True,
    )

# Update layout for consistent styling
fig.update_layout(
    title_text=f'Europe Cumulative CO2 Emissions vs. Annual Average Land Temperature ({start_plot_year}-{end_plot_year})',
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    title_font_color='white',
    legend_title_font_color='white',
    hovermode='x unified',
    margin=dict(t=80, l=50, r=50, b=100) # Adjust margins
)

# Update x-axis
fig.update_xaxes(
    title_text='Year',
    showgrid=True,
    gridcolor='rgba(255,255,255,0.2)',
    tickfont_color='white',
    title_font_color='white',
    showline=True, linewidth=2, linecolor='white'
)

# Update primary y-axis (Cumulative CO2)
fig.update_yaxes(
    title_text='Cumulative CO2 Emissions (Mt)',
    secondary_y=False,
    showgrid=True,
    gridcolor='rgba(255,255,255,0.2)',
    tickfont_color='white',
    title_font_color='white',
    showline=True, linewidth=2, linecolor='white'
)

# Update secondary y-axis (Annual Average Temperature)
fig.update_yaxes(
    title_text='Temperature in degrees Celsius', # Changed label as requested
    secondary_y=True,
    showgrid=False,
    tickfont_color='white',
    title_font_color='white',
    showline=True, linewidth=2, linecolor='white'
)

# Add your annotation a little lower (change y to -0.25 if you want it even further down)
fig.add_annotation(
    text=(
        "Click on a Europe Cumalative CO2/CO2 Trendline etc to select/deselect line from the graph.<br>"
        "Zoom in by right clicking a square.<br>"
    ),
    xref="paper", yref="paper",
    x=-0.05, y=-0.25,
    showarrow=False,
    font=dict(color='white', size=10),
    align='left'
)



fig.show()

The graph shows two rising curves: one for cumulative CO2, the other for annual average land temperatures. As emissions climb across the 20th and early 21st centuries, temperatures rise in parallel—nearly 2°C over the full span. This correlation underscores the stakes of Europe’s energy shift: it’s not just about modernizing infrastructure, but preventing the environmental destabilization already underway.

**Summary and outlook**

In conclusion, Europe’s energy transition is shaped by two parallel yet distinct forces: the economic challenge of restructuring its energy systems, and the climate imperative to cut emissions fast. These forces are connected — economic delays slow climate action — but they also stand alone as serious threats.

The economic reality involves massive investments, volatile energy prices, and uneven capacity across countries. It’s a political and logistical challenge shaped by infrastructure, markets, and inequality. Meanwhile, the climate crisis operates on a different level: rising temperatures and ongoing emissions, driven by physical systems that won’t wait for financial readiness.

Though both problems demand urgent attention, they differ in nature. One is about human systems and policy coordination; the other about irreversible environmental consequences. Addressing both is non-negotiable. Europe must act decisively on each front, ensuring the transition is not just possible, but fast enough to matter.


## References

1. Saraji, Z. et al. (2023). Urgency and justice bring enabling and jeopardizing dynamics to energy transitions. Energy Research & Social Science. https://www.sciencedirect.com/science/article/pii/S2210422423000734?utm

2. Schyska, B. U. & Kies, A. (2019). How regional differences in cost of capital influence the optimal design of power systems. arXiv. https://arxiv.org/abs/1903.04768?utm

3. Victoria, M., Brown, T. et al. (2020). Early decarbonisation of the European energy system pays off. arXiv. https://arxiv.org/abs/2004.11009?utm

4. Wyszomierski, R. et al. (2025). The Cost-Effectiveness of Renewable Energy Sources in the European Union’s Ecological Economic Framework. Sustainability. https://www.mdpi.com/2071-1050/17/10/4715?utm

5. Apeti, A. E., Bambe, B. W. W., Edoh, E. D., & Ly, A. (2025). Wealth inequality and carbon inequality. Ecological Economics, 227. https://www.sciencedirect.com/science/article/abs/pii/S0921800924003033

6. Cevik, S., Ninomiya, K., (2022). Chasing the Sun and Catching the Wind: Energy Transition and Electricity Prices in Europe. https://www.elibrary.imf.org/view/journals/001/2022/220/article-A001-en.xml

7. European Commission. (2020). The European Green Deal. European Commission. https://commission.europa.eu/strategy-and-policy/priorities-2019-2024/european-green-deal_en

8. IRENA (2021). Renewable Energy Statistics 2021. https://pcreee.org/publication/irena-renewable-energy-statistics-2021